In [1]:
import os 
files = os.listdir(
    "paciente 1"
)

In [39]:
import ollama
import numpy as np
from datetime import datetime
import pymupdf
import json
from codecarbon import OfflineEmissionsTracker
from icecream import ic

In [17]:
files[1]

'2_Informe evolución post trasplante_EVENTO.pdf'

In [18]:
doc = pymupdf.open(f"paciente 1/{files[0]}") # open a document
out = open(f"output_{files[0]}.txt", "wb") # create a text output
tables_all = []
for page in doc: # iterate the document pages
    text = page.get_text().encode("latin-1") # get plain text (is in UTF-8)
    out.write(text) # write text of page
    out.write(bytes((12,))) # write page delimiter (form feed 0x0C)
out.close()

In [19]:
def AskOllama(prompt, model_name = "gpt-oss:latest"):
    from ollama import chat
    from ollama import ChatResponse

    response: ChatResponse = chat(model=model_name, messages=[
    {
        'role': 'user',
        'content': prompt,
    },
    ])

    return response['message']['content']

In [20]:
def TranslateOllama(text, code_input, code_output, model = "translategemma:4b"):

    language = {
        "es": "Spanish",
        "en": "English", 

    }

    SOURCE_LANG = language[code_input]
    TARGET_LANG = language[code_output]
    prompt = f"""You are a professional {SOURCE_LANG} ({code_input}) to {TARGET_LANG} ({code_output}) translator. Your goal is to accurately convey the meaning and nuances of the original {SOURCE_LANG} text while adhering to {TARGET_LANG} grammar, vocabulary, and cultural sensitivities.
    Produce only the {TARGET_LANG} translation, without any additional explanations or commentary. Please translate the following {SOURCE_LANG} text into {TARGET_LANG}:


    {text}"""

    response = AskOllama(prompt, model_name= model)

    return response

In [21]:
def IdentifyOllama(text, model = "translategemma:4b"):

    language = {
        "es": "Spanish",
        "en": "English", 

    }
    prompt = f"""You are a language expert. Tell me the language of the following text:

    {text}.
    
    The possible languages are:

    {language}

    RESPOND ONLY WITH THE LANGUAGE CODE. FOR INSTANCE, IF IT IS IN SPANISH RETURN 'ES'. 

    DON'T ADD ANYTHING ELSE.

    """

    response = AskOllama(prompt, model_name= model)

    return response

# TEXT ANALYSIS

In [28]:
language_input = IdentifyOllama("Whatcha gonna do when the chips are down, now that the chips are down?").strip()

In [29]:
language_output = "es"

In [30]:
TranslateOllama("Whatcha gonna do when the chips are down, now that the chips are down?", language_input, language_output)

'¿Qué vas a hacer cuando las cosas se pongan difíciles, ahora que ya lo están?'

In [33]:
with open(f"output_{files[0]}.txt", "r", encoding="latin-1") as f:
    text_ = f.read()

In [41]:
tracker = OfflineEmissionsTracker(project_name="Test Ollama", measure_power_secs=0.1)
tracker.start()
models_checked = []
for model in ollama.list()["models"]:
    prompt = f"""

    Del siguiente texto: {text_}

    Sácame como diccionario la siguiente información: 

    - Fecha de la consulta: Fecha en formato '%Y-%m-%d %H:%M:%S'
    - Motivo de la consulta: Breve resumen
    - Datos de la consulta: Por ejemplo, si hay una exploración, sacar los datos físicos de la misma.En caso de que no haya ningún dato devolver un diccionario vacío
    - Información extra: Basándote en los análisis, sácame algun valor que esté fuera de rango (si los hay)

    DEVUELVE SÓLO ESA INFORMACIÓN
    
    """


    size = np.round(model["size"]/(1000*1000*1000),2)
    ic(model["model"])
    ic(size)

    if size < 10 and model["model"] in "medgemma:4b":

        tracker.start_task(f"Ask {model['model']}")
        for i in range(3):
            try:
                response = AskOllama(prompt=prompt, model_name = model["model"])

                dict_ = json.loads(response.replace("python", "").replace("json", "").replace("```",""))
                
                result_filename = f"result_paciente1_{files[0]}_{model['model'].replace(':','_')}_{i}.json"
                
                with open(result_filename, "w") as file:
                    json.dump(dict_,file)
                
                models_checked.append({
                    "model_name": model["model"],
                    "file": result_filename
                })
            except Exception as e:
                print(e)

        tracker.stop_task()
    
emissions = tracker.stop()


[codecarbon INFO @ 17:02:29] offline tracker init
[codecarbon WARNING @ 17:02:29] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon WARNING @ 17:02:29] Error while trying to count physical CPUs: [Errno 2] No such file or directory: 'lscpu'. Defaulting to 1.
[codecarbon INFO @ 17:02:29] [setup] RAM Tracking...
[codecarbon INFO @ 17:02:29] [setup] CPU Tracking...
[codecarbon WARNING @ 17:02:29] We saw that you have a Apple M5 Max but we don't know it. Please contact us.
[codecarbon WARNING @ 17:02:29] We will use the default power consumption of 4 W per thread for your 18 CPU, so 72W.
[codecarbon WARNING @ 17:02:29] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Mac OS and ARM processor detected: Please enable PowerMetrics sudo to measure CPU

[codecarbon INFO @ 17:02:29] CPU Model on constant consumption mode: Apple M5 Max
[codecarbon WARNING @ 17:02:29] No CPU tracking mode found. Falling back on CPU load mode.
[codecarbon

In [42]:
models_checked

[{'model_name': 'medgemma:4b',
  'file': 'result_paciente1_5_informe microbiologia resolucion evento.pdf_medgemma_4b_0.json'},
 {'model_name': 'medgemma:4b',
  'file': 'result_paciente1_5_informe microbiologia resolucion evento.pdf_medgemma_4b_1.json'},
 {'model_name': 'medgemma:4b',
  'file': 'result_paciente1_5_informe microbiologia resolucion evento.pdf_medgemma_4b_2.json'}]

In [43]:
with open(f"result_{model['model'].replace(':','_')}_{i}.json", "w") as file:
    json.dump(dict_,file)

In [44]:
f"paciente_1_{model['model']}_{i}.json"

'paciente_1_granite4.1:8b_2.json'

In [45]:


print(f"Emissions : {1000 * emissions} g CO₂")
for task_name, task in tracker._tasks.items():
    print(
        f"Emissions : {1000 * task.emissions_data.emissions} g CO₂ for task {task_name}"
    )
    print(
        f"Energy consumed : { task.emissions_data.energy_consumed} kWh for task {task_name}"
    )
    print(
        f"Duration : { task.emissions_data.duration} (s) for task {task_name}"
    )

Emissions : 0.02144786480372272 g CO₂
Emissions : 0.02141163639236577 g CO₂ for task Ask medgemma:4b
Energy consumed : 4.507712924708583e-05 kWh for task Ask medgemma:4b
Duration : 11.711477375007235 (s) for task Ask medgemma:4b
